<a href="https://colab.research.google.com/github/Mariam-Elbishbeashy/HeadlineGeneration-NLP/blob/main/headlineGenerationNLP_flan_t5_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📰 FLAN-T5-Base News Headline Generation Model

This project implements an end-to-end training pipeline for automatic news headline generation using the :contentReference[oaicite:0]{index=0}. The system fine-tunes a large encoder–decoder transformer to convert news articles into concise, high-quality headlines.

---

## Model Architecture

The system is based on the :contentReference[oaicite:1]{index=1}, which follows a **text-to-text transformer framework**.

- **Architecture:** Encoder–Decoder Transformer  
- **Model Size:** Base variant of T5 (~220M parameters)  
- **Framework:** Hugging Face Transformers  
- **Task:** Abstractive Headline Generation  

The model is optimized using gradient checkpointing to reduce memory usage during training.

---

## Training Strategy

The model is fine-tuned using supervised learning on paired data:

- **Input:** News article  
- **Target:** Human-written headline  
- **Trainer:** `Seq2SeqTrainer`

### Key Training Settings
- Learning Rate: `2e-5`
- Epochs: `3`
- Batch Size: `1` (with gradient accumulation = 8)
- Optimizer: AdamW (default Trainer optimizer)
- Weight Decay: `0.01`
- Mixed Precision: Disabled (FP16 = False for stability)
- Best Model Selection: Based on `ROUGE-L`

Gradient accumulation is used to simulate a larger batch size while saving GPU memory.

---

## Evaluation Method

Model performance is evaluated using **ROUGE metrics**:

- **ROUGE-1:** Word overlap accuracy  
- **ROUGE-2:** Phrase overlap  
- **ROUGE-L:** Sequence similarity (main selection metric)

The model automatically keeps the best checkpoint based on ROUGE-L score.

---

## Optimization & Efficiency Techniques

To handle memory constraints and improve training stability:

- Gradient Checkpointing enabled  
- CUDA cache clearing and garbage collection used  
- Small batch size with accumulation  
- Early stopping (patience = 2 epochs)  
- Best model loading at end of training  

These techniques allow efficient training of :contentReference[oaicite:2]{index=2} on limited GPU resources.

---

## Inference Pipeline

After training, the model generates headlines as follows:

1. Input article is tokenized  
2. Passed through encoder–decoder model  
3. Beam search decoding (num_beams = 2)  
4. Output tokens decoded into text  

### Example Behavior:
**Input:**  
News article text about Apple AI features  

**Output:**  
Generated concise headline describing the event

---

## Model Saving

The trained model is saved to Google Drive:

- Final model weights  
- Tokenizer files  
- Configuration files  

This allows reuse for inference or deployment without retraining.



In [ ]:


!pip -q install -U transformers datasets evaluate accelerate sentencepiece rouge_score



import os
import gc
import torch
import pandas as pd
import numpy as np
import evaluate

from google.colab import drive
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)


os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


if torch.cuda.is_available():

    device = torch.device("cuda")

    print("=" * 60)
    print("USING GPU:", torch.cuda.get_device_name(0))
    print("=" * 60)

else:

    device = torch.device("cpu")

    print("=" * 60)
    print("USING CPU")
    print("=" * 60)


drive.mount('/content/drive')


DATA_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/news_headline_model_data"

TRAIN_PATH = os.path.join(DATA_DIR, "flan_t5_base_train.csv")
VAL_PATH   = os.path.join(DATA_DIR, "flan_t5_base_validation.csv")
TEST_PATH  = os.path.join(DATA_DIR, "flan_t5_base_test.csv")

OUTPUT_DIR = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_base_model"

os.makedirs(OUTPUT_DIR, exist_ok=True)


print("=" * 60)
print("LOADING DATASETS")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

train_df = train_df[["model_input", "model_target"]].dropna()
val_df   = val_df[["model_input", "model_target"]].dropna()
test_df  = test_df[["model_input", "model_target"]].dropna()

print("Train Samples:", len(train_df))
print("Validation Samples:", len(val_df))
print("Test Samples:", len(test_df))


train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)


MODEL_NAME = "google/flan-t5-base"

print("=" * 60)
print("LOADING MODEL")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    low_cpu_mem_usage=True,
    tie_word_embeddings=False,
)

model.gradient_checkpointing_enable()

model.to(device)

print("MODEL LOADED SUCCESSFULLY")


MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 32

def tokenize_function(batch):

    model_inputs = tokenizer(
        batch["model_input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=batch["model_target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

print("=" * 60)
print("TOKENIZING DATASETS")
print("=" * 60)

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

print("TOKENIZATION COMPLETE")


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8,
)


rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    result = {
        key: round(value * 100, 4)
        for key, value in result.items()
    }

    return result


training_args = Seq2SeqTrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    learning_rate=2e-5,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,

    metric_for_best_model="rougeL",

    greater_is_better=True,

    predict_with_generate=True,

    generation_max_length=MAX_TARGET_LENGTH,

    generation_num_beams=2,

    logging_steps=100,

    report_to="none",

    fp16=False,

    bf16=False,

    remove_unused_columns=True,

    dataloader_num_workers=2,

    seed=42,
)


print("=" * 60)
print("INITIALIZING TRAINER")
print("=" * 60)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_tokenized,

    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

print("TRAINER READY")


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("=" * 60)
print("START TRAINING")
print("=" * 60)

trainer.train()


print("=" * 60)
print("EVALUATING MODEL")
print("=" * 60)

results = trainer.evaluate(test_tokenized)

print(results)


FINAL_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "final_model"
)

trainer.save_model(FINAL_MODEL_PATH)

tokenizer.save_pretrained(FINAL_MODEL_PATH)

print("=" * 60)
print("MODEL SAVED")
print(FINAL_MODEL_PATH)
print("=" * 60)


def generate_headline(article):

    model.eval()

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=2,
        )

    headline = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return headline


sample_article = """
Apple introduced new AI features for Siri and
smart notification summaries during its latest
developer conference.
"""

headline = generate_headline(sample_article)

print("\nARTICLE:")
print(sample_article)

print("\nGENERATED HEADLINE:")
print(headline)


print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 35.1 MB/s eta 0:00:00
USING GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOADING DATASETS
Train Samples: 38076
Validation Samples: 4760
Test Samples: 4760
LOADING MODEL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
0.000000,nan,3,0.000000,0.000000,0.000000,0.000000


{'eval_loss': nan, 'eval_rouge1': 0.0, 'eval_rouge2': 0.0, 'eval_rougeL': 0.0, 'eval_rougeLsum': 0.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL SAVED
/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_base_model/final_model

ARTICLE:

Apple introduced new AI features for Siri and
smart notification summaries during its latest
developer conference.


GENERATED HEADLINE:

TRAINING COMPLETE


---

## Summary

This pipeline demonstrates a full production-style NLP workflow using the :contentReference[oaicite:3]{index=3}, including:

- Efficient training with memory optimization  
- ROUGE-based evaluation  
- Early stopping for better generalization  
- Real-time headline generation  

The final system successfully converts long news articles into short, meaningful headlines using a fine-tuned transformer model.

# VERIFY MODEL SAVED CORRECTLY


In [ ]:


print("=" * 60)
print("VERIFYING SAVED MODEL FILES")
print("=" * 60)

saved_files = os.listdir(FINAL_MODEL_PATH)

print("Saved files:")
for f in saved_files:
    print(" -", f)

required_files = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "spiece.model"
]

print("\nCHECKING REQUIRED FILES:")
for rf in required_files:
    if rf in saved_files:
        print(f" OK {rf}")
    else:
        print(f" MISSING {rf}")

VERIFYING SAVED MODEL FILES
Saved files:
 - config.json
 - generation_config.json
 - model.safetensors
 - tokenizer_config.json
 - tokenizer.json
 - training_args.bin

CHECKING REQUIRED FILES:
 OK config.json
 OK model.safetensors
 OK tokenizer_config.json
 MISSING spiece.model


# REAL-TIME INFERENCE MODE


In [ ]:


print("=" * 60)
print("REAL-TIME HEADLINE GENERATION MODE")
print("Type 'exit' to stop")
print("=" * 60)

while True:

    user_input = input("\nEnter article text:\n")

    if user_input.lower() == "exit":
        print("Exiting real-time mode...")
        break

    inputs = tokenizer(
        user_input,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=2,
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\n🧠 Generated Headline:")
    print(prediction)
    print("-" * 60)

REAL-TIME HEADLINE GENERATION MODE
Type 'exit' to stop

Enter article text:
Apple launches new AI chip for faster iPhone performance


[transformers] Both `max_new_tokens` (=32) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🧠 Generated Headline:

------------------------------------------------------------

Enter article text:
exit
Exiting real-time mode...


#LOAD MODELS

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


MODEL_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_base_model/final_model"


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)


model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)


device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

print("MODEL LOADED SUCCESSFULLY")
print("DEVICE:", device)

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


MODEL LOADED SUCCESSFULLY
DEVICE: cuda


# FLAN-T5-BASE FINAL EVALUATION


In [ ]:


import torch
import numpy as np
import pandas as pd
import evaluate

from tqdm import tqdm
from transformers import DataCollatorForSeq2Seq


rouge = evaluate.load("rouge")


device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

print("=" * 60)
print("EVALUATION DEVICE:", device)
print("=" * 60)


data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt"
)


predictions = []
references = []


batch_size = 8


print("=" * 60)
print("STARTING TEST EVALUATION")
print("=" * 60)

with torch.no_grad():

    for i in tqdm(
        range(0, len(test_tokenized), batch_size)
    ):


        batch_samples = [
            test_tokenized[j]
            for j in range(
                i,
                min(i + batch_size, len(test_tokenized))
            )
        ]


        batch = data_collator(batch_samples)

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        outputs = model.generate(

            input_ids=input_ids,

            attention_mask=attention_mask,

            max_length=32,

            num_beams=4,

            early_stopping=True
        )


        preds = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )


        labels = batch["labels"].cpu().numpy()

        labels = np.where(
            labels != -100,
            labels,
            tokenizer.pad_token_id
        )

        refs = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        predictions.extend(preds)
        references.extend(refs)


results = rouge.compute(

    predictions=predictions,

    references=references,

    use_stemmer=True
)


print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

for key, value in results.items():

    print(f"{key}: {round(value * 100, 4)}")

print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for i in range(5):

    print(f"\nEXAMPLE {i+1}")
    print("-" * 50)

    print("GENERATED:")
    print(predictions[i])

    print("\nREFERENCE:")
    print(references[i])

results_df = pd.DataFrame({

    "Generated Headline": predictions,

    "Reference Headline": references
})

SAVE_PATH = "/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_base_model/final_test_predictions.csv"

results_df.to_csv(
    SAVE_PATH,
    index=False
)

print("\n" + "=" * 60)
print("PREDICTIONS SAVED")
print(SAVE_PATH)
print("=" * 60)

EVALUATION DEVICE: cuda
STARTING TEST EVALUATION


100%|██████████| 595/595 [01:01<00:00,  9.71it/s]



FINAL TEST RESULTS
rouge1: 0.0
rouge2: 0.0
rougeL: 0.0
rougeLsum: 0.0

SAMPLE PREDICTIONS

EXAMPLE 1
--------------------------------------------------
GENERATED:
,

REFERENCE:
Opinion | Anita Hill: Class Actions Could Fight Discrimination in Tech

EXAMPLE 2
--------------------------------------------------
GENERATED:
,

REFERENCE:
KHL Player Executes Sliding Dab That is Delightfully Reminiscent of This Kid

EXAMPLE 3
--------------------------------------------------
GENERATED:
,

REFERENCE:
Statkraft, partners to build Europe's largest onshore wind farm for 1.1 bln euros

EXAMPLE 4
--------------------------------------------------
GENERATED:
,

REFERENCE:
Muniz wins World Surfing Games gold, Olympic hosts Japan claim first medals

EXAMPLE 5
--------------------------------------------------
GENERATED:
,

REFERENCE:
Huge asteroid to pass close to Earth

PREDICTIONS SAVED
/content/drive/MyDrive/headlineGenerationProjectNLP/flan_t5_base_model/final_test_predictions.csv
